# Known-development syntax comparison

Compare automatic proposals with 42 AI-assisted owner-confirmed development selections. Thirty historical focus positions remain proposed. Exact alignment only; no forced matching, independent accuracy or reference changes.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import pandas as pd
from IPython.display import display
current = Path.cwd()
ROOT = next((p for p in [current, *current.parents] if (p / "src/compare_confirmed_syntax_v1.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Start from the project directory or notebooks folder.")
syntax_python = Path(os.environ.get("SYNTAX_PYTHON", str(ROOT / ".venv/syntax-prototype/bin/python")))
if not syntax_python.exists():
    raise FileNotFoundError("Create the optional syntax environment as described in docs/SYNTACTIC_CANDIDATES_v1.md.")
subprocess.run([str(syntax_python), str(ROOT / "src/compare_confirmed_syntax_v1.py")], check=True)
report = json.loads((ROOT / "data/processed/confirmed_syntax_comparison_v1.json").read_text())
summary = report["summary"]
print("Alignment:", summary["alignment_counts"])
print("Aligned focus confirmation:", summary["aligned_focus_confirmation"])
display(pd.DataFrame(summary["field_relations_all_exact"]).fillna(0).astype(int))


In [ ]:
rows = []
for item in report["comparisons"]:
    if item["alignment"] != "exact_offsets":
        rows.append({"claim_id": item["claim_id"], "execution": item["execution_number"],
                     "focus": item["predicate_focus"], "issue": "no_exact_alignment",
                     "field": "", "reference": "", "automatic": ""})
    else:
        for field, relation in item["field_relations"].items():
            if relation == "definite_value_conflict":
                rows.append({"claim_id": item["claim_id"], "execution": item["execution_number"],
                    "focus": item["predicate_focus"], "issue": relation, "field": field,
                    "reference": item["reference_fields"][field],
                    "automatic": item["automatic_proposal"]["proposed_fields"][field]})
display(pd.DataFrame(rows))
print("Diagnostic differences only. Unclear outputs are reported separately above.")
